# Vision pipelines

[Back to the introduction](ch01_demo.ipynb). Run in order. `LLM.pipeline(task, model=...)` also loads Transformers.js vision tasks: image classification, zero-shot image classification, image features and object detection. Each section loads a small browser model, runs it on two images, and releases it before the next one. Image tasks accept an HTTP(S) URL, encoded JPEG/PNG/WebP bytes, a file in the JupyterLite file browser, a `uint8` NumPy array or a PIL image. Encoded bytes are the cheapest to send to the browser. The native backend does not run vision tasks.

In [ ]:
from IPython.display import Image, display

from lmlite import LLM

base = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main"
images = {"tiger": f"{base}/tiger.jpg", "cats": f"{base}/cats.jpg"}
for url in images.values():
    display(Image(url=url, width=240))

## Image classification

A ViT model scores the 1,000 ImageNet classes. `top_k` limits the labels returned.

In [ ]:
classifier = await LLM.pipeline("image-classification", model="vit-base-patch16-224", backend="wasm")
for name, url in images.items():
    print(name, [(item["label"], round(item["score"], 3)) for item in await classifier(url, top_k=3)])
await classifier.dispose()

## Zero-shot image classification

CLIP ranks labels you choose at call time. Phrases such as "a photo of ..." work better than single words.

In [ ]:
clip = await LLM.pipeline("zero-shot-image-classification", model="clip-vit-base-patch32", backend="wasm")
labels = ["a photo of a tiger", "a photo of cats", "a photo of food"]
for name, url in images.items():
    result = await clip(url, candidate_labels=labels)
    print(name, [(item["label"], round(item["score"], 3)) for item in result])
await clip.dispose()

## Image features

DINOv2 turns each image into one vector (the first, CLS, token). Pass a list to embed several images in one call, then compare them with cosine similarity.

In [ ]:
import math

extractor = await LLM.pipeline("image-feature-extraction", model="dinov2-small", backend="wasm")
tiger, cats = await extractor(list(images.values()))
dot = sum(a * b for a, b in zip(tiger, cats))
print(len(tiger), "dimensions; cosine similarity:", round(dot / math.hypot(*tiger) / math.hypot(*cats), 3))
await extractor.dispose()

## Object detection

DETR returns boxes in pixels (`xmin`, `ymin`, `xmax`, `ymax`) unless you pass `percentage=True`. Raise `threshold` to keep only confident detections.

In [ ]:
detector = await LLM.pipeline("object-detection", model="detr-resnet-50", backend="wasm")
for item in await detector(images["cats"], threshold=0.9):
    box = item["box"]
    print(item["label"], round(item["score"], 3), [round(box[key]) for key in ("xmin", "ymin", "xmax", "ymax")])
await detector.dispose()

The inference pipelines are released; cached files are retained for subsequent runs. See the usage guide for task options and output formats.